# C-MET 中文完整复现：数据、训练、推理与评估

这不是只生成两个示例视频的 demo，而是 C-MET **主方法**的完整可执行链路：

```text
原始 MEAD / CREMA-D
-> 官方 EDTalk 裁脸、256x256、25 FPS、16 kHz 音频
-> emotion2vec+large 与 EDTalk 特征
-> 严格数据门禁
-> 训练缓存
-> 2 step 冒烟训练
-> 20 万 step 主实验与两组论文损失消融
-> 基础/扩展情绪定性推理
-> MEAD / CREMA-D 官方测试清单推理
-> 技术检查与论文指标汇总接口
```

> [!IMPORTANT]
> 论文训练硬件是单张 RTX 3090 24GB。A100 40GB 足够；完整训练建议使用 A100 和高内存运行时。所有长任务默认关闭，必须由你显式打开开关。

> [!CAUTION]
> 官方没有公开精确的 FID/FVD/SyncNet 采样实现、微调后的 Emotion-FAN、统一 baseline 环境、PD-FGC 完整分支、Qwen2.5-Omni 分支、连续编辑脚本和用户研究数据。因此，本笔记本完整执行主方法链路，但不会把自定义接口冒充成作者未公开的精确论文协议。


## 使用顺序

本 Notebook 有两条入口，**不要混着运行**：

### 路线 A：推荐的一键数据预处理

只运行下面的“▶ 一键完成公开数据预处理”单元。它已经包含 Drive 挂载、仓库克隆、依赖安装、官方代码固定、MEAD/CREMA-D smoke 和完整预处理。

一键单元运行期间，**不要继续点击后面的 Drive、克隆、依赖和数据预处理单元**。这些单元是为后续特征抽取、训练以及手动排错保留的。

### 路线 B：数据完成后的特征、训练、推理与评估

当一键单元最终显示 `下一阶段: features` 后：

1. 从“数据预处理完成后再继续”开始，依次运行 0～8，初始化当前 Python 会话并下载训练/推理权重。
2. 跳过 9～12 的手动数据预处理区，因为一键单元已经完成这些工作。
3. 从 13 开始进行 emotion2vec、EDTalk 特征抽取、门禁、训练缓存和训练。
4. 先跑 2 step 冒烟训练，再启动 20 万 step 主实验；Colab 中断后使用 `RESUME_*` 续训。
5. 最后运行定性结果、MEAD/CREMA-D benchmark 和指标汇总。

重复执行 0～8 中的 Drive 挂载、Git 更新和依赖检查是幂等的，不会重新处理完整数据。


## ▶ 一键完成公开数据预处理（推荐）

只运行下一格即可，不需要打开 Colab 终端，也不需要手工输入 Git 或 Bash 命令。该单元会自动：

```text
挂载 Google Drive
→ 克隆或更新复现仓库
→ 固定并修补官方 C-MET
→ 安装数据预处理依赖
→ MEAD 来源预检与 smoke
→ CREMA-D smoke
→ MEAD / CREMA-D 完整预处理与断点续跑
```

> 运行这一格时，请停在这里，不要继续运行下面的单元。下面的 Drive、克隆、依赖和数据单元属于后续完整复现/手动模式，功能会与本格部分重复。

> 仍需你在弹出的 Google Drive 授权窗口确认一次，并事先把有权限访问的 MEAD 视频 Part 快捷方式放进 `MyDrive/MEAD`。这是 Google 账号授权，代码无法替你点击。

> 完整预处理耗时较长。Colab 断线后重新打开 Notebook，再运行同一格即可自动续跑。


In [ ]:
# @title ▶ 点击运行：自动完成 MEAD / CREMA-D 数据预处理
AUTO_CONTINUE_FULL = True # @param {type:"boolean"}

import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive


def run(command, *, cwd=None, env=None):
    command = [str(value) for value in command]
    print("$", shlex.join(command), flush=True)
    subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
    )


DRIVE_MOUNT = Path("/content/drive")
MY_DRIVE = DRIVE_MOUNT / "MyDrive"
MEAD_SHARED_ROOT = MY_DRIVE / "MEAD"
REPRO_ROOT = Path("/content/cmet-repro-colab")
CMET_ROOT = Path("/content/C-MET")
REPRO_URL = "https://github.com/ChengyangHe-ux/cmet-repro-colab.git"
OFFICIAL_URL = "https://github.com/ChanHyeok-Choi/C-MET.git"
OFFICIAL_COMMIT = "0ca437cf7a8129c6a5dca1e2667a588410822bbe"

print("== 1/6 挂载 Google Drive ==", flush=True)
drive.mount(str(DRIVE_MOUNT), force_remount=False)
if not MY_DRIVE.is_dir():
    raise RuntimeError("Google Drive 没有正确挂载到 /content/drive/MyDrive。")
if not MEAD_SHARED_ROOT.is_dir():
    raise FileNotFoundError(
        "没有找到 MyDrive/MEAD。请先在 Google Drive 中建立 MEAD 聚合目录，"
        "并把有权限访问的官方视频 Part 快捷方式放进去。"
    )
print("Drive 与 MEAD 聚合目录检查通过：", MEAD_SHARED_ROOT, flush=True)

print("== 2/6 克隆或更新复现仓库 ==", flush=True)
if REPRO_ROOT.exists() and not (REPRO_ROOT / ".git").is_dir():
    shutil.rmtree(REPRO_ROOT)
if not (REPRO_ROOT / ".git").is_dir():
    run(["git", "clone", REPRO_URL, REPRO_ROOT])
else:
    run(["git", "-C", REPRO_ROOT, "pull", "--ff-only"])
print(
    "复现工具 commit:",
    subprocess.check_output(
        ["git", "-C", str(REPRO_ROOT), "rev-parse", "--short", "HEAD"],
        text=True,
    ).strip(),
    flush=True,
)

print("== 3/6 安装系统与 Python 依赖 ==", flush=True)
missing_commands = [
    name for name in ["ffmpeg", "pkg-config", "git-lfs"]
    if shutil.which(name) is None
]
if missing_commands:
    run(["apt-get", "update", "-y"])
    run(["apt-get", "install", "-y", "ffmpeg", "pkg-config", "git-lfs"])
else:
    print("系统依赖已经存在，跳过 apt 安装。", flush=True)
run([
    sys.executable,
    REPRO_ROOT / "scripts" / "install_colab_dependencies.py",
    "--requirements",
    REPRO_ROOT / "configs" / "colab_requirements.txt",
])

print("== 4/6 准备并固定官方 C-MET ==", flush=True)
if CMET_ROOT.exists() and not (CMET_ROOT / ".git").is_dir():
    shutil.rmtree(CMET_ROOT)
if not (CMET_ROOT / ".git").is_dir():
    run(["git", "clone", OFFICIAL_URL, CMET_ROOT])
run(["git", "-C", CMET_ROOT, "fetch", "origin", OFFICIAL_COMMIT])
run(["git", "-C", CMET_ROOT, "reset", "--hard", OFFICIAL_COMMIT])
run([
    sys.executable,
    REPRO_ROOT / "scripts" / "patch_cmet_colab_full.py",
    "--cmet-root",
    CMET_ROOT,
])

print("== 5/6 启动可恢复数据流水线 ==", flush=True)
environment = os.environ.copy()
environment.update(
    {
        "REPRO_ROOT": str(REPRO_ROOT),
        "CMET_ROOT": str(CMET_ROOT),
        "MEAD_SHARED_ROOT": str(MEAD_SHARED_ROOT),
        "OUT_ROOT": str(MY_DRIVE / "C-MET-full"),
        "CMET_CONTINUE_FULL": "1" if AUTO_CONTINUE_FULL else "0",
        "PYTHONUNBUFFERED": "1",
    }
)
run(
    ["bash", REPRO_ROOT / "scripts" / "colab_resume_data_pipeline.sh"],
    cwd=REPRO_ROOT,
    env=environment,
)

print("== 6/6 数据流水线本次运行结束 ==", flush=True)
print(
    "状态报告：",
    MY_DRIVE / "C-MET-full" / "reports",
    flush=True,
)


## 数据预处理完成后再继续：初始化特征与训练环境

只有当顶部一键单元显示 `下一阶段: features` 后，才从这里继续。

下面 0～8 会重新确认 Drive、路径、仓库、依赖、补丁和权重。部分操作看起来与一键单元重复，但它们用于把后续特征与训练所需的变量、函数和权重加载到当前 Python 会话中；不会重新执行完整数据预处理。

### 0. 检查 GPU

完整流程不支持 CPU。论文使用 24GB RTX 3090；这里把 20GB 作为最低门禁，A100 40GB 更稳。


In [ ]:
import json
import os
import platform
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA 可用:", torch.cuda.is_available())
subprocess.run(["nvidia-smi"], check=False)

if not torch.cuda.is_available():
    raise RuntimeError("没有检测到 CUDA。请在 Colab 的运行时设置中选择 GPU 后重新连接。")

GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {GPU_NAME}；显存: {VRAM_GB:.2f} GB")
if VRAM_GB < 20:
    raise RuntimeError("显存低于 20GB，不建议继续完整训练。请切换到 A100/L4/24GB 以上 GPU。")


### 1. 挂载并验证 Google Drive

顶部一键单元已经在当前运行时挂载过 Drive 时，本格通常会直接复用现有挂载。保留本格是为了在 Colab 重连或新运行时中重新建立后续单元需要的 `MY_DRIVE` 等变量。

如果出现 `credential propagation was unsuccessful`：

1. 浏览器只保留一个 Google 账号登录。
2. 允许第三方 Cookie、弹窗和重定向。
3. 断开并删除当前 Colab 运行时，再重新连接。
4. 重新运行本格，不要继续创建 `/content/drive/MyDrive` 目录。

挂载失败时本格会立即停止，防止把临时磁盘误当成 Drive。


In [ ]:
from google.colab import drive

DRIVE_MOUNT = Path("/content/drive")
try:
    drive.mount(str(DRIVE_MOUNT), force_remount=False)
except Exception as exc:
    print("Drive 授权失败。请按上一格的四个步骤处理后重试。")
    raise RuntimeError("Google Drive 挂载失败，已阻止后续步骤。") from exc

mounts = set()
mount_file = Path("/proc/mounts")
if mount_file.is_file():
    for row in mount_file.read_text(encoding="utf-8", errors="ignore").splitlines():
        parts = row.split()
        if len(parts) >= 2:
            mounts.add(parts[1].replace("\\040", " "))

MY_DRIVE = DRIVE_MOUNT / "MyDrive"
real_mount = MY_DRIVE.is_dir() and (os.path.ismount(DRIVE_MOUNT) or str(DRIVE_MOUNT) in mounts)
if not real_mount:
    raise RuntimeError("/content/drive 没有真实挂载。不要手动创建伪 MyDrive 目录，请重新授权。")

probe = MY_DRIVE / ".cmet_drive_write_test"
probe.write_text("ok\n", encoding="utf-8")
probe.unlink()
print("Drive 已真实挂载且可写:", MY_DRIVE)


## 2. 设置路径与实验参数

默认 `DATA_SOURCE="public"`。`MyDrive/MEAD` 是官方快捷方式聚合根目录，可直接包含身份目录，也可保留 `Part*/身份` 结构；如果你放在别处，只修改 `MEAD_SHARED_ROOT`。已经自行下载完整原始数据时，也可改成 `manual` 并配置 `RAW_MEAD`、`RAW_CREMAD`。其余产物统一放到 `MyDrive/C-MET-full`。


In [ ]:
DRIVE_ROOT = MY_DRIVE / "C-MET-full"
RAW_ROOT = DRIVE_ROOT / "raw"
RAW_MEAD = RAW_ROOT / "MEAD"
RAW_CREMAD = RAW_ROOT / "CREMA-D"
MEAD_PUBLIC_URL = "https://drive.google.com/drive/folders/1GwXP-KpWOxOenOxITTsURJZQ_1pkd4-j"
MEAD_SHARED_ROOT = MY_DRIVE / "MEAD"
PUBLIC_DATA_WORK_ROOT = Path("/content/cmet_public_data")

PROCESSED_ROOT = DRIVE_ROOT / "dataset"
MEAD_DRIVE = PROCESSED_ROOT / "MEAD" / "FPS25"
CREMAD_DRIVE = PROCESSED_ROOT / "CREMA_D" / "FPS25"
MODEL_ROOT = DRIVE_ROOT / "official_model_files"
RUNS_ROOT = DRIVE_ROOT / "reproduction_runs"
QUALITATIVE_ROOT = DRIVE_ROOT / "qualitative_runs"
BENCHMARK_ROOT = DRIVE_ROOT / "benchmark_runs"
REPORT_ROOT = DRIVE_ROOT / "reports"
CACHE_ARCHIVE = DRIVE_ROOT / "cache" / "cmet_training_cache.tar"

LOCAL_CACHE_ROOT = Path("/content/cmet_training_data")
LOCAL_MEAD = LOCAL_CACHE_ROOT / "MEAD" / "FPS25"
CMET_ROOT = Path("/content/C-MET")
REPRO_ROOT = Path("/content/cmet-repro-colab")

OFFICIAL_COMMIT = "0ca437cf7a8129c6a5dca1e2667a588410822bbe"
DATA_SOURCE = "public"  # 可选："public" 或 "manual"
SEED = 42
NUM_SAMPLES = 10
EXPRESSION_BATCH_SIZE = 128
RENDER_BATCH_SIZE = 8
CROP_MODE = "official"  # 原始视频已稳定裁好脸时才改成 "none"
CHECKPOINT_SOURCE = "self_trained"  # 可选："self_trained" 或 "official"
BENCHMARK_EMOTION_PROTOCOL = "dataset"  # 可选："dataset" 或 "official-static"
ALLOW_PARTIAL_SELF_TRAINED_CHECKPOINT = False
DOWNLOAD_EDTALK_V = False  # 只有官方视频唇形分支或 subprocess 排错才需要

if DATA_SOURCE not in {"public", "manual"}:
    raise ValueError("DATA_SOURCE 只能是 public 或 manual。")

for path in [
    DRIVE_ROOT,
    RAW_ROOT,
    MEAD_DRIVE,
    CREMAD_DRIVE,
    MODEL_ROOT,
    RUNS_ROOT,
    QUALITATIVE_ROOT,
    BENCHMARK_ROOT,
    REPORT_ROOT,
    CACHE_ARCHIVE.parent,
]:
    path.mkdir(parents=True, exist_ok=True)

usage = shutil.disk_usage(DRIVE_ROOT)
print("复现根目录:", DRIVE_ROOT)
print("数据来源模式:", DATA_SOURCE)
print("MEAD 官方快捷方式:", MEAD_SHARED_ROOT)
print("手工 MEAD 原始数据备用路径:", RAW_MEAD)
print("手工 CREMA-D 原始数据备用路径:", RAW_CREMAD)
print(f"Drive 报告的可用空间: {usage.free / 1024**3:.1f} GB")
print(f"Colab 本地盘可用空间: {shutil.disk_usage('/content').free / 1024**3:.1f} GB")


## 3. 长任务开关

所有耗时任务默认是 `False`。每次只打开当前要跑的一项，运行完成后改回 `False`，避免“全部运行”时误触发长任务。


In [ ]:
RUN_MEAD_SOURCE_CHECK = False
RUN_MEAD_PREP_SMOKE = False
RUN_CREMAD_PREP_SMOKE = False
RUN_MEAD_PREP_FULL = False
RUN_CREMAD_PREP_FULL = False
RUN_FULL_MEDIA_GATE = False

RUN_MEAD_E2V_SMOKE = False
RUN_CREMAD_E2V_SMOKE = False
RUN_MEAD_E2V_FULL = False
RUN_CREMAD_E2V_FULL = False
RUN_EDTALK_SMOKE = False
RUN_EDTALK_FULL = False
RUN_FEATURE_GATE = False

BUILD_TRAIN_CACHE = False
EXTRACT_TRAIN_CACHE = False
RUN_LOCAL_CACHE_GATE = False

RUN_TRAIN_SMOKE = False
RESUME_TRAIN_SMOKE = False
RUN_MAIN_TRAIN = False
RESUME_MAIN_TRAIN = False
RUN_ABLATION_RECON_ONLY = False
RESUME_ABLATION_RECON_ONLY = False
RUN_ABLATION_RECON_CNT = False
RESUME_ABLATION_RECON_CNT = False

RUN_QUAL_BASIC = False
RUN_QUAL_EXTENDED = False
RUN_QUAL_TECH_CHECK = False

RUN_MEAD_BENCH_SMOKE = False
RUN_CREMAD_BENCH_SMOKE = False
RUN_MEAD_BENCH_FULL = False
RUN_CREMAD_BENCH_FULL = False
RUN_BENCH_TECH_CHECK = False

RUN_PARTIAL_METRIC_SUMMARY = False
RUN_STRICT_PAPER_METRICS = False


## 4. 公共执行函数

所有外部命令都会先打印完整命令，失败时立即抛错，不会静默继续。


In [ ]:
def run_command(command, cwd=None):
    command = [str(value) for value in command]
    print("$", shlex.join(command))
    subprocess.run(command, cwd=str(cwd) if cwd else None, check=True)


def run_repro_script(script_name, *arguments):
    run_command([sys.executable, REPRO_ROOT / "scripts" / script_name, *arguments])


def require_file(path, label):
    if path is None:
        raise FileNotFoundError(f"{label} 尚未生成或尚未选择。")
    path = Path(path)
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f"{label} 缺失或为空：{path}")
    return path


def require_directory(path, label):
    path = Path(path)
    if not path.is_dir():
        raise FileNotFoundError(f"{label} 目录不存在：{path}")
    return path


### 5. 克隆或更新仓库并固定官方版本

这一步与一键单元中的仓库准备部分重复，但采用“存在则更新、不存在才克隆”的方式，不会重复下载完整数据。保留它是为了让后续特征、训练和推理单元在新的 Colab 运行时中也能独立恢复。

官方 C-MET 固定到 `0ca437c`。复现脚本从你的 GitHub 仓库 `main` 获取。


In [ ]:
OFFICIAL_URL = "https://github.com/ChanHyeok-Choi/C-MET.git"
REPRO_URL = "https://github.com/ChengyangHe-ux/cmet-repro-colab.git"

if not REPRO_ROOT.exists():
    run_command(["git", "clone", REPRO_URL, REPRO_ROOT])
elif not (REPRO_ROOT / ".git").is_dir():
    raise RuntimeError(f"路径已存在但不是 Git 仓库：{REPRO_ROOT}")
else:
    run_command(["git", "-C", REPRO_ROOT, "pull", "--ff-only"])

if not CMET_ROOT.exists():
    run_command(["git", "clone", OFFICIAL_URL, CMET_ROOT])
elif not (CMET_ROOT / ".git").is_dir():
    raise RuntimeError(f"路径已存在但不是 Git 仓库：{CMET_ROOT}")

current_commit = subprocess.check_output(
    ["git", "-C", str(CMET_ROOT), "rev-parse", "HEAD"], text=True
).strip()
if current_commit != OFFICIAL_COMMIT:
    dirty = subprocess.check_output(
        ["git", "-C", str(CMET_ROOT), "status", "--porcelain"], text=True
    ).strip()
    if dirty:
        raise RuntimeError("官方仓库版本不对且已有修改。请重启运行时后重新运行，避免覆盖文件。")
    run_command(["git", "-C", CMET_ROOT, "fetch", "origin", OFFICIAL_COMMIT])
    run_command(["git", "-C", CMET_ROOT, "switch", "--detach", OFFICIAL_COMMIT])

actual_commit = subprocess.check_output(
    ["git", "-C", str(CMET_ROOT), "rev-parse", "HEAD"], text=True
).strip()
if actual_commit != OFFICIAL_COMMIT:
    raise RuntimeError(f"官方 commit 固定失败：{actual_commit}")

print("官方 C-MET commit:", actual_commit)
print("复现工具 commit:", subprocess.check_output(["git", "-C", str(REPRO_ROOT), "rev-parse", "HEAD"], text=True).strip())


### 6. 确认系统与 Python 依赖

这一步会再次检查依赖；已经安装且版本满足时会复用，不会触发数据预处理。安装脚本**不会升级 pip，也不会重装 Torch/CUDA**。它固定 `setuptools<82` 和 `pandas==2.2.2`，避免此前 Colab 环境冲突。


In [ ]:
run_command(["apt-get", "update", "-y"])
run_command(["apt-get", "install", "-y", "ffmpeg", "pkg-config", "git-lfs"])
run_repro_script(
    "install_colab_dependencies.py",
    "--requirements",
    REPRO_ROOT / "configs" / "colab_requirements.txt",
)
print("依赖安装完成。下一格必须执行补丁和环境自检。")


### 7. 打兼容补丁、连接 Drive、下载后续权重

兼容补丁是幂等的，重复运行不会叠加修改。这里还会建立后续特征/训练所需的 Drive 软链接并下载官方权重，因此数据预处理完成后仍需运行一次。

补丁只处理 Colab/Python/PyTorch 兼容、路径、checkpoint 和可恢复训练，不改 C-MET 模型结构与论文损失。


In [ ]:
run_repro_script("patch_cmet_colab_full.py", "--cmet-root", CMET_ROOT)


def link_directory(link_path, target_path):
    link_path = Path(link_path)
    target_path = Path(target_path)
    target_path.mkdir(parents=True, exist_ok=True)
    link_path.parent.mkdir(parents=True, exist_ok=True)
    if link_path.is_symlink():
        if link_path.resolve() == target_path.resolve():
            print("软链接已存在:", link_path, "->", target_path)
            return
        link_path.unlink()
    elif link_path.exists():
        if link_path.is_dir() and not any(link_path.iterdir()):
            link_path.rmdir()
        else:
            raise RuntimeError(f"不会覆盖已有非空路径：{link_path}")
    os.symlink(target_path, link_path, target_is_directory=True)
    print("已连接:", link_path, "->", target_path)


link_directory(CMET_ROOT / "dataset" / "MEAD" / "FPS25", MEAD_DRIVE)
link_directory(CMET_ROOT / "dataset" / "CREMA_D" / "FPS25", CREMAD_DRIVE)
link_directory(CMET_ROOT / "pretrained_weights", MODEL_ROOT / "pretrained_weights")
link_directory(CMET_ROOT / "checkpoints", MODEL_ROOT / "checkpoints")

weight_command = [
    sys.executable,
    REPRO_ROOT / "scripts" / "download_pretrained_weights.py",
    "--output-root",
    MODEL_ROOT,
    "--include-connector",
    "--report",
    REPORT_ROOT / "weight_download.json",
]
if DOWNLOAD_EDTALK_V:
    weight_command.append("--include-edtalk-v")
run_command(weight_command)


## 8. 环境硬门禁

必须看到 `"ready": true` 才能继续。它会检查真实 Drive、GPU、显存、固定 commit、依赖版本、ffmpeg、git-lfs 和核心权重。


In [ ]:
run_repro_script(
    "verify_colab_environment.py",
    "--cmet-root",
    CMET_ROOT,
    "--drive-root",
    DRIVE_ROOT,
    "--expected-commit",
    OFFICIAL_COMMIT,
    "--min-vram-gb",
    "20",
    "--require-file",
    MODEL_ROOT / "pretrained_weights" / "Audio2Lip.pt",
    "--require-file",
    MODEL_ROOT / "pretrained_weights" / "EDTalk.pt",
    "--require-file",
    MODEL_ROOT / "checkpoints" / "_epoch_2105_checkpoint_step000200000.pth",
    "--report",
    REPORT_ROOT / "colab_environment.json",
)


## 手动数据模式（使用顶部一键单元后跳过 9～12）

### 9. 数据来源与 MEAD 快捷方式预检

如果顶部一键数据预处理已经成功完成，请直接跳到第 13 节，不要重新打开这里的数据开关。

本区仅用于手动排错或希望逐阶段控制数据流程时使用。默认公开来源模式只需要一次人工授权动作：

1. 打开 [MEAD 官方公共文件夹](https://drive.google.com/drive/folders/1GwXP-KpWOxOenOxITTsURJZQ_1pkd4-j)。
2. 对你有权访问的视频 Part 或身份目录选择“整理” -> “添加快捷方式”。
3. 在“我的云端硬盘”建立 `MEAD` 聚合目录，可保留 `Part0/身份`、`Part1/身份` 结构。
4. 回到这里，把 `RUN_MEAD_SOURCE_CHECK=True` 后重新运行开关格和本格。

预检会一次性确认官方 47 个身份对应的 `video.tar` 或 `video_*.tar` 分卷都能访问，支持顶层身份和向下 3 层的 Part 目录，并会一次列出所有缺失身份；预检不会复制或处理数据。如果某个身份目录只有音频/标注而没有视频 tar，必须补充包含该身份视频的官方 Part。CREMA-D 不需要手工上传，后面的脚本通过官方 Git LFS 镜像只下载 C-MET benchmark 实际引用的文件。

> `manual` 模式仍支持你自行取得的原始数据，并保留原目录和文件名。


In [ ]:
VIDEO_SUFFIXES = {".flv", ".mkv", ".mov", ".mp4", ".webm"}


def count_raw_videos(root):
    root = Path(root)
    if not root.is_dir():
        return 0
    return sum(path.suffix.lower() in VIDEO_SUFFIXES for path in root.rglob("*"))


def prepare_manual_dataset(dataset, raw_root, out_root, report, limit=None):
    command = [
        sys.executable,
        REPRO_ROOT / "scripts" / "prepare_datasets.py",
        "--dataset",
        dataset,
        "--raw-root",
        raw_root,
        "--out-root",
        out_root,
        "--cmet-root",
        CMET_ROOT,
        "--crop-mode",
        CROP_MODE,
        "--report",
        report,
    ]
    if dataset == "mead":
        command.extend(
            [
                "--mead-subset",
                "official",
                "--speaker-file",
                CMET_ROOT / "dataset" / "MEAD" / "train.txt",
                "--speaker-file",
                CMET_ROOT / "dataset" / "MEAD" / "test.txt",
            ]
        )
    if limit is not None:
        command.extend(["--limit", str(limit)])
    run_command(command)


def prepare_public_mead(check_only=False, limit_identities=None, limit_videos=None, keep_work=False):
    if CROP_MODE != "official":
        raise ValueError("公开 MEAD 流程固定使用官方 EDTalk 裁脸，请保持 CROP_MODE='official'。")
    command = [
        sys.executable,
        REPRO_ROOT / "scripts" / "prepare_public_datasets.py",
        "mead",
        "--shared-root",
        MEAD_SHARED_ROOT,
        "--cmet-root",
        CMET_ROOT,
        "--out-root",
        MEAD_DRIVE,
        "--work-root",
        PUBLIC_DATA_WORK_ROOT,
        "--report-root",
        REPORT_ROOT,
    ]
    if check_only:
        command.append("--check-only")
    if limit_identities is not None:
        command.extend(["--limit-identities", str(limit_identities)])
    if limit_videos is not None:
        command.extend(["--limit-videos", str(limit_videos)])
    if keep_work:
        command.append("--keep-work")
    run_command(command)


def prepare_public_cremad(limit_videos=None, cleanup_source=False):
    command = [
        sys.executable,
        REPRO_ROOT / "scripts" / "prepare_public_datasets.py",
        "crema-d",
        "--cmet-root",
        CMET_ROOT,
        "--out-root",
        CREMAD_DRIVE,
        "--work-root",
        PUBLIC_DATA_WORK_ROOT,
        "--report-root",
        REPORT_ROOT,
    ]
    if limit_videos is not None:
        command.extend(["--limit-videos", str(limit_videos)])
    if cleanup_source:
        command.append("--cleanup-source")
    run_command(command)


if DATA_SOURCE == "public":
    print("MEAD 官方快捷方式应位于:", MEAD_SHARED_ROOT)
    print("当前是否可见:", MEAD_SHARED_ROOT.is_dir())
    if RUN_MEAD_SOURCE_CHECK:
        prepare_public_mead(check_only=True)
    else:
        print("来源预检未启用。添加快捷方式后，把 RUN_MEAD_SOURCE_CHECK 改成 True。")
    print("CREMA-D 将从官方 Git LFS 镜像自动按清单下载，无需上传原始仓库。")
else:
    print("MEAD 原始视频数:", count_raw_videos(RAW_MEAD), "目录:", RAW_MEAD)
    print("CREMA-D 原始视频数:", count_raw_videos(RAW_CREMAD), "目录:", RAW_CREMAD)
    print("如果这里是 0，请先把已获授权的数据放到对应 Drive 目录。")


### 10. 数据预处理 smoke test（手动模式）

使用顶部一键单元后跳过本节。手动模式下，来源预检通过后，把对应 smoke 开关改成 `True`。每个数据集只处理 2 条视频，验证下载、解包、官方裁脸、25 FPS、256x256 和 16 kHz WAV 全链路。


In [ ]:
if DATA_SOURCE == "public":
    if RUN_MEAD_PREP_SMOKE:
        prepare_public_mead(limit_identities=1, limit_videos=2, keep_work=True)
    if RUN_CREMAD_PREP_SMOKE:
        prepare_public_cremad(limit_videos=2)
else:
    if RUN_MEAD_PREP_SMOKE:
        prepare_manual_dataset("mead", RAW_MEAD, MEAD_DRIVE, REPORT_ROOT / "prepare_mead_smoke.json", limit=2)
    if RUN_CREMAD_PREP_SMOKE:
        prepare_manual_dataset("crema-d", RAW_CREMAD, CREMAD_DRIVE, REPORT_ROOT / "prepare_cremad_smoke.json", limit=2)
if not RUN_MEAD_PREP_SMOKE and not RUN_CREMAD_PREP_SMOKE:
    print("smoke 预处理未启用。")


In [ ]:
def inspect_prepare_report(path):
    path = Path(path)
    if not path.is_file():
        print("尚无报告:", path)
        return
    value = json.loads(path.read_text(encoding="utf-8"))
    print(path.name, value.get("counts", value.get("status")))
    failures = [row for row in value.get("results", []) if row.get("status") == "failed"]
    for row in failures[:3]:
        print("失败示例:", row)


if DATA_SOURCE == "public":
    inspect_prepare_report(REPORT_ROOT / "mead_public_stream_state.json")
    inspect_prepare_report(REPORT_ROOT / "prepare_cremad_public_smoke.json")
else:
    inspect_prepare_report(REPORT_ROOT / "prepare_mead_smoke.json")
    inspect_prepare_report(REPORT_ROOT / "prepare_cremad_smoke.json")


### 11. 完整数据预处理（手动模式）

使用顶部一键单元后跳过本节。手动模式下，smoke 成功后再打开 full 开关。所有状态都会写回 Drive，断线后重跑同一 full 开关即可继续。


In [ ]:
if DATA_SOURCE == "public":
    if RUN_MEAD_PREP_FULL:
        prepare_public_mead()
    if RUN_CREMAD_PREP_FULL:
        prepare_public_cremad(cleanup_source=True)
else:
    if RUN_MEAD_PREP_FULL:
        prepare_manual_dataset("mead", RAW_MEAD, MEAD_DRIVE, REPORT_ROOT / "prepare_mead_full.json")
    if RUN_CREMAD_PREP_FULL:
        prepare_manual_dataset("crema-d", RAW_CREMAD, CREMAD_DRIVE, REPORT_ROOT / "prepare_cremad_full.json")
if not RUN_MEAD_PREP_FULL and not RUN_CREMAD_PREP_FULL:
    print("完整预处理未启用。")
else:
    drive_usage = shutil.disk_usage(DRIVE_ROOT)
    local_usage = shutil.disk_usage('/content')
    print(f"数据预处理后 Drive 可用: {drive_usage.free / 1024**3:.1f} GB")
    print(f"数据预处理后本地盘可用: {local_usage.free / 1024**3:.1f} GB")


### 12. 完整媒体门禁（手动模式）

顶部一键流程完成后，媒体状态已经由数据流水线记录；正常情况下可直接进入第 13 节。需要单独复核全部 MP4/WAV 时，才打开 `RUN_FULL_MEDIA_GATE`。


In [ ]:
if RUN_FULL_MEDIA_GATE:
    run_repro_script(
        "validate_full_dataset.py",
        "--cmet-root",
        CMET_ROOT,
        "--mead-root",
        MEAD_DRIVE,
        "--cremad-root",
        CREMAD_DRIVE,
        "--scope",
        "train",
        "--split",
        "both",
        "--stage",
        "media",
        "--check-media",
        "--strict",
        "--report",
        REPORT_ROOT / "train_media_gate.json",
    )
    run_repro_script(
        "validate_full_dataset.py",
        "--cmet-root",
        CMET_ROOT,
        "--mead-root",
        MEAD_DRIVE,
        "--cremad-root",
        CREMAD_DRIVE,
        "--scope",
        "benchmark",
        "--benchmark-dataset",
        "both",
        "--stage",
        "media",
        "--check-media",
        "--strict",
        "--report",
        REPORT_ROOT / "benchmark_media_gate.json",
    )
else:
    print("完整媒体门禁未启用。")


## 一键数据预处理成功后的正式下一步

### 13. 抽取 MEAD 与 CREMA-D 的 emotion2vec+large 特征

从这里开始不再重复数据预处理。emotion2vec 会把每段音频转换成 1024 维情绪语义特征。smoke 每个数据集只处理前 2 个 WAV；full 遍历完整数据。MEAD 特征用于训练和 benchmark，CREMA-D 特征用于强度感知 benchmark。已有且 shape 为 `(1024,)`、不含 NaN/Inf 的特征会自动跳过。


In [ ]:
def extract_e2v(data_root, limit, report):
    command = [
        sys.executable,
        REPRO_ROOT / "scripts" / "extract_emotion2vec_features.py",
        "--data-root",
        data_root,
        "--device",
        "cuda",
        "--report",
        report,
    ]
    if limit is not None:
        command.extend(["--limit", str(limit)])
    run_command(command)


if RUN_MEAD_E2V_SMOKE:
    extract_e2v(MEAD_DRIVE, 2, REPORT_ROOT / "emotion2vec_mead_smoke.json")
if RUN_CREMAD_E2V_SMOKE:
    extract_e2v(CREMAD_DRIVE, 2, REPORT_ROOT / "emotion2vec_cremad_smoke.json")
if RUN_MEAD_E2V_FULL:
    extract_e2v(MEAD_DRIVE, None, REPORT_ROOT / "emotion2vec_mead_full.json")
if RUN_CREMAD_E2V_FULL:
    extract_e2v(CREMAD_DRIVE, None, REPORT_ROOT / "emotion2vec_cremad_full.json")
if not any([RUN_MEAD_E2V_SMOKE, RUN_CREMAD_E2V_SMOKE, RUN_MEAD_E2V_FULL, RUN_CREMAD_E2V_FULL]):
    print("emotion2vec 特征抽取未启用。")


## 14. 抽取 EDTalk 表情、姿态、唇形特征

smoke 只处理 2 个视频。`--batch-size` 是单视频内部的帧批大小；A100 40GB 默认 100，显存不足时可改为 32 或 16。


In [ ]:
def extract_edtalk(limit, report):
    command = [
        sys.executable,
        REPRO_ROOT / "scripts" / "extract_edtalk_features.py",
        "--cmet-root",
        CMET_ROOT,
        "--data-root",
        MEAD_DRIVE,
        "--batch-size",
        "100",
        "--report",
        report,
    ]
    if limit is not None:
        command.extend(["--limit", str(limit)])
    run_command(command)


if RUN_EDTALK_SMOKE:
    extract_edtalk(2, REPORT_ROOT / "edtalk_smoke.json")
if RUN_EDTALK_FULL:
    extract_edtalk(None, REPORT_ROOT / "edtalk_full.json")
if not RUN_EDTALK_SMOKE and not RUN_EDTALK_FULL:
    print("EDTalk 特征抽取未启用。")


## 15. 特征与模型合同门禁

`training` 检查完整 MEAD MP4/WAV/三类 EDTalk/e2v；`model` 再按官方 DataLoader 实际读取的编号子集检查；`benchmark features` 检查 MEAD/CREMA-D 测试清单中全部源/目标语音的 emotion2vec。三份报告都显示 `ready: true` 才进入训练与 benchmark。


In [ ]:
if RUN_FEATURE_GATE:
    for stage in ["training", "model"]:
        run_repro_script(
            "validate_full_dataset.py",
            "--cmet-root",
            CMET_ROOT,
            "--mead-root",
            MEAD_DRIVE,
            "--scope",
            "train",
            "--split",
            "both",
            "--stage",
            stage,
            "--check-arrays",
            "--strict",
            "--report",
            REPORT_ROOT / f"feature_gate_{stage}.json",
        )
    run_repro_script(
        "validate_full_dataset.py",
        "--cmet-root",
        CMET_ROOT,
        "--mead-root",
        MEAD_DRIVE,
        "--cremad-root",
        CREMAD_DRIVE,
        "--scope",
        "benchmark",
        "--benchmark-dataset",
        "both",
        "--stage",
        "features",
        "--check-arrays",
        "--strict",
        "--report",
        REPORT_ROOT / "feature_gate_benchmark.json",
    )
else:
    print("特征门禁未启用。")


## 16. 构建并解压训练缓存

直接从 Drive 读取数万个小 NPY 会很慢。缓存只打包训练实际读取的 `*_ED_exp.npy`、emotion2vec NPY 和零字节 MP4 路径标记；官方训练不会解码 MP4。主数据仍保留在 Drive，训练时解压到 `/content`。


In [ ]:
if BUILD_TRAIN_CACHE:
    run_repro_script(
        "build_training_cache.py",
        "--mead-root",
        MEAD_DRIVE,
        "--output",
        CACHE_ARCHIVE,
        "--manifest",
        REPORT_ROOT / "training_cache.json",
    )
else:
    print("训练缓存构建未启用。")


In [ ]:
import tarfile

if EXTRACT_TRAIN_CACHE:
    require_file(CACHE_ARCHIVE, "训练缓存")
    if LOCAL_CACHE_ROOT.exists() and any(LOCAL_CACHE_ROOT.iterdir()):
        raise RuntimeError(
            f"本地缓存目录已非空：{LOCAL_CACHE_ROOT}。为避免混合旧数据，请重启运行时后再解压。"
        )
    LOCAL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    free_bytes = shutil.disk_usage("/content").free
    if free_bytes < CACHE_ARCHIVE.stat().st_size * 1.2:
        raise RuntimeError("Colab 本地磁盘空间不足，无法安全解压训练缓存。")
    with tarfile.open(CACHE_ARCHIVE, "r") as archive:
        try:
            archive.extractall(LOCAL_CACHE_ROOT, filter="data")
        except TypeError:
            archive.extractall(LOCAL_CACHE_ROOT)
    print("训练缓存已解压:", LOCAL_MEAD)
else:
    print("训练缓存解压未启用。每次新运行时都需要重新解压。")


In [ ]:
if RUN_LOCAL_CACHE_GATE:
    run_repro_script(
        "validate_full_dataset.py",
        "--cmet-root",
        CMET_ROOT,
        "--mead-root",
        LOCAL_MEAD,
        "--scope",
        "train",
        "--split",
        "both",
        "--stage",
        "model",
        "--check-arrays",
        "--strict",
        "--report",
        REPORT_ROOT / "local_training_cache_gate.json",
    )
else:
    print("本地训练缓存门禁未启用。")


## 17. 训练命令 dry-run

这一步不加载数据、不训练，只生成实验 manifest，确认主实验与两组论文损失消融都固定为 20 万 step；smoke 为 2 step、1 个验证 batch、每步保存 checkpoint。


In [ ]:
for experiment, smoke in [
    ("paper_main", True),
    ("paper_main", False),
    ("ablation_recon_only", False),
    ("ablation_recon_cnt", False),
]:
    command = [
        sys.executable,
        REPRO_ROOT / "scripts" / "run_training.py",
        "--cmet-root",
        CMET_ROOT,
        "--dataset-root",
        LOCAL_MEAD,
        "--experiment",
        experiment,
        "--output-root",
        RUNS_ROOT / "dry_run_checks",
        "--skip-patch",
        "--skip-validation",
        "--dry-run",
    ]
    if smoke:
        command.append("--smoke")
    else:
        command.extend(["--checkpoint-interval", "1000", "--evaluate-interval", "5000"])
    run_command(command)


### 学习练习：读懂实验 manifest

运行下一格，比较主实验和两组消融的 `lambda_cnt`、`lambda_dir` 与 `max_steps`。你应该能解释：

- `ablation_recon_only` 为什么两个 lambda 都是 0；
- `ablation_recon_cnt` 为什么只保留 `lambda_cnt=0.1`；
- 三组为什么都必须训练到 20 万 step 才是公平比较。


In [ ]:
for name in [
    f"paper_main_seed{SEED}",
    f"ablation_recon_only_seed{SEED}",
    f"ablation_recon_cnt_seed{SEED}",
]:
    path = RUNS_ROOT / "dry_run_checks" / "manifests" / f"{name}.json"
    value = json.loads(path.read_text(encoding="utf-8"))
    experiment = value["experiment"]
    command = value["command"]
    max_steps = command[command.index("--max_steps") + 1]
    print(
        name,
        "lambda_cnt=", experiment["lambda_cnt"],
        "lambda_dir=", experiment["lambda_dir"],
        "max_steps=", max_steps,
        "checkpoint_interval=", command[command.index("--checkpoint_interval") + 1],
        "keep_recent=", command[command.index("--checkpoint_keep_recent") + 1],
        "milestone_interval=", command[command.index("--checkpoint_milestone_interval") + 1],
        "evaluate_interval=", command[command.index("--evaluate_interval") + 1],
    )


## 18. 2 step 冒烟训练

先打开 `RUN_TRAIN_SMOKE`。如果同名 smoke checkpoint 已存在，改为 `RESUME_TRAIN_SMOKE=True`。成功标准是完成训练、验证并生成 step 1/2 checkpoint。


In [ ]:
if RUN_TRAIN_SMOKE and RESUME_TRAIN_SMOKE:
    raise ValueError("RUN_TRAIN_SMOKE 和 RESUME_TRAIN_SMOKE 不能同时为 True。")

if RUN_TRAIN_SMOKE or RESUME_TRAIN_SMOKE:
    require_directory(LOCAL_MEAD, "本地 MEAD 训练缓存")
    command = [
        sys.executable,
        REPRO_ROOT / "scripts" / "run_training.py",
        "--cmet-root",
        CMET_ROOT,
        "--dataset-root",
        LOCAL_MEAD,
        "--experiment",
        "paper_main",
        "--output-root",
        RUNS_ROOT,
        "--smoke",
        "--skip-patch",
        "--validation-stage",
        "model",
    ]
    if RESUME_TRAIN_SMOKE:
        command.extend(["--resume", "auto"])
    run_command(command)
else:
    print("冒烟训练未启用。")


## 19. 20 万 step 主实验与自动续训

首次运行打开 `RUN_MAIN_TRAIN`；Colab 中断后重新完成前置步骤、解压缓存，再打开 `RESUME_MAIN_TRAIN`。checkpoint 每 1000 step 写回 Drive，验证仍每 5000 step 执行；新 checkpoint 原子写入成功后，只保留最近 3 份和每 5 万 step 里程碑，避免写满 Drive。

续训会恢复模型、优化器、step 和主进程随机状态；由于多进程 DataLoader worker 状态未被官方代码保存，不能声称中断前后逐 bit 完全一致。


In [ ]:
if RUN_MAIN_TRAIN and RESUME_MAIN_TRAIN:
    raise ValueError("RUN_MAIN_TRAIN 和 RESUME_MAIN_TRAIN 不能同时为 True。")

if RUN_MAIN_TRAIN or RESUME_MAIN_TRAIN:
    require_directory(LOCAL_MEAD, "本地 MEAD 训练缓存")
    command = [
        sys.executable,
        REPRO_ROOT / "scripts" / "run_training.py",
        "--cmet-root",
        CMET_ROOT,
        "--dataset-root",
        LOCAL_MEAD,
        "--experiment",
        "paper_main",
        "--output-root",
        RUNS_ROOT,
        "--checkpoint-interval",
        "1000",
        "--evaluate-interval",
        "5000",
        "--skip-patch",
        "--validation-stage",
        "model",
    ]
    if RESUME_MAIN_TRAIN:
        command.extend(["--resume", "auto"])
    run_command(command)
else:
    print("主实验训练未启用。")


In [ ]:
from tensorboard import notebook as tensorboard_notebook

print("TensorBoard 日志目录:", RUNS_ROOT / "tensorboard")
print("需要查看曲线时，取消下一行注释并运行本格。")
# tensorboard_notebook.start(f"--logdir {RUNS_ROOT / 'tensorboard'}")


## 20. 论文 Table 6 的两组损失消融

这里只运行论文明确对应的两组：`Lrecon` 和 `Lrecon + Lcnt`。每组都有独立 manifest、TensorBoard 和 checkpoint，不覆盖主实验，并使用与主实验相同的 checkpoint 保留策略。


In [ ]:
def run_training_experiment(experiment, start_flag, resume_flag):
    if start_flag and resume_flag:
        raise ValueError(f"{experiment} 的首次训练和续训开关不能同时为 True。")
    if not start_flag and not resume_flag:
        print(experiment, "未启用")
        return
    require_directory(LOCAL_MEAD, "本地 MEAD 训练缓存")
    command = [
        sys.executable,
        REPRO_ROOT / "scripts" / "run_training.py",
        "--cmet-root",
        CMET_ROOT,
        "--dataset-root",
        LOCAL_MEAD,
        "--experiment",
        experiment,
        "--output-root",
        RUNS_ROOT,
        "--checkpoint-interval",
        "1000",
        "--evaluate-interval",
        "5000",
        "--skip-patch",
        "--validation-stage",
        "model",
    ]
    if resume_flag:
        command.extend(["--resume", "auto"])
    run_command(command)


run_training_experiment(
    "ablation_recon_only",
    RUN_ABLATION_RECON_ONLY,
    RESUME_ABLATION_RECON_ONLY,
)
run_training_experiment(
    "ablation_recon_cnt",
    RUN_ABLATION_RECON_CNT,
    RESUME_ABLATION_RECON_CNT,
)


## 21. 选择推理 checkpoint

完整复现应使用 `self_trained`。如果训练尚未完成、只想验证推理环境，可暂时把配置改为 `official`；汇报时必须明确区分官方权重结果和自训练结果。


In [ ]:
def checkpoint_step(path):
    try:
        return int(Path(path).stem.rsplit("step", 1)[1])
    except (IndexError, ValueError):
        return -1


def latest_checkpoint(run_name):
    directory = RUNS_ROOT / "checkpoints" / run_name
    values = (
        [path for path in directory.glob("*_checkpoint_step*.pth") if path.is_file() and path.stat().st_size > 0]
        if directory.is_dir()
        else []
    )
    return max(values, key=checkpoint_step) if values else None


OFFICIAL_CHECKPOINT = MODEL_ROOT / "checkpoints" / "_epoch_2105_checkpoint_step000200000.pth"
SELF_TRAINED_CHECKPOINT = latest_checkpoint(f"paper_main_seed{SEED}")

if CHECKPOINT_SOURCE == "official":
    SELECTED_CHECKPOINT = OFFICIAL_CHECKPOINT
elif CHECKPOINT_SOURCE == "self_trained":
    SELECTED_CHECKPOINT = SELF_TRAINED_CHECKPOINT
else:
    raise ValueError("CHECKPOINT_SOURCE 只能是 self_trained 或 official。")

print("官方 checkpoint:", OFFICIAL_CHECKPOINT)
print("最新自训练 checkpoint:", SELF_TRAINED_CHECKPOINT)
print("当前选择:", SELECTED_CHECKPOINT)
CHECKPOINT_TAG = SELECTED_CHECKPOINT.stem if SELECTED_CHECKPOINT is not None else "checkpoint_not_selected"
QUALITATIVE_OUTPUT = QUALITATIVE_ROOT / CHECKPOINT_SOURCE / CHECKPOINT_TAG
BENCHMARK_OUTPUT = BENCHMARK_ROOT / CHECKPOINT_SOURCE / CHECKPOINT_TAG
print("结果隔离标签:", CHECKPOINT_TAG)


def require_selected_checkpoint():
    checkpoint = require_file(SELECTED_CHECKPOINT, "推理 checkpoint")
    if (
        CHECKPOINT_SOURCE == "self_trained"
        and checkpoint_step(checkpoint) < 200000
        and not ALLOW_PARTIAL_SELF_TRAINED_CHECKPOINT
    ):
        raise RuntimeError(
            f"自训练 checkpoint 只有 step {checkpoint_step(checkpoint)}，尚未达到论文的 200000 step。"
            "如只做中间诊断，可显式设置 ALLOW_PARTIAL_SELF_TRAINED_CHECKPOINT=True。"
        )
    return checkpoint


## 22. 基础情绪与扩展情绪定性推理

每组只加载一次模型；进度写入 JSONL，重连后会跳过已完成情绪。基础情绪 7 类，扩展情绪 6 类，neutral 始终读取官方 `audios/MEAD/neutral` 特征池。


In [ ]:
def run_qualitative(group):
    checkpoint = require_selected_checkpoint()
    run_repro_script(
        "run_qualitative_inference.py",
        "--cmet-root",
        CMET_ROOT,
        "--checkpoint",
        checkpoint,
        "--output-root",
        QUALITATIVE_OUTPUT,
        "--group",
        group,
        "--num-samples",
        str(NUM_SAMPLES),
        "--seed",
        str(SEED),
        "--device",
        "cuda",
        "--expression-batch-size",
        str(EXPRESSION_BATCH_SIZE),
        "--render-batch-size",
        str(RENDER_BATCH_SIZE),
        "--fail-fast",
    )


if RUN_QUAL_BASIC:
    run_qualitative("basic")
if RUN_QUAL_EXTENDED:
    run_qualitative("extended")
if not RUN_QUAL_BASIC and not RUN_QUAL_EXTENDED:
    print("定性推理未启用。")


In [ ]:
if RUN_QUAL_TECH_CHECK:
    run_repro_script(
        "evaluate_videos_basic.py",
        "--video-dir",
        QUALITATIVE_OUTPUT,
        "--out-csv",
        REPORT_ROOT / f"qualitative_{CHECKPOINT_TAG}_technical.csv",
        "--report",
        REPORT_ROOT / f"qualitative_{CHECKPOINT_TAG}_technical.json",
        "--strict",
    )
else:
    print("定性视频技术检查未启用。")


## 23. MEAD 与 CREMA-D 官方测试清单推理

- MEAD：1143 条；CREMA-D：1546 条。
- smoke 各跑 2 条；full 会读取相同 JSONL 并自动跳过 smoke 已完成样本。
- 默认 `dataset` 协议按 benchmark 自身的情绪与强度构建 10-shot 语音特征；`official-static` 只复用官方发布的 level_3 演示池。
- persistent 后端只加载一次模型，并在加载前预检所选输入、权重和全部 emotion2vec 特征。

> [!WARNING]
> 作者没有公开 Table 1 的批量推理脚本。`dataset` 是依据论文 10 speech-shots、`test.csv` 强度字段和官方 `Dataset.get_raw_e2v` 重建的协议；`official-static` 是可执行对照。两者都会记录在 manifest 中，未经作者确认都不能称为精确作者协议。


In [ ]:
def run_benchmark(dataset, limit=None):
    checkpoint = require_selected_checkpoint()
    command = [
        sys.executable,
        REPRO_ROOT / "scripts" / "run_benchmark_inference.py",
        "--cmet-root",
        CMET_ROOT,
        "--dataset",
        dataset,
        "--checkpoint",
        checkpoint,
        "--output-root",
        BENCHMARK_OUTPUT,
        "--emotion-protocol",
        BENCHMARK_EMOTION_PROTOCOL,
        "--feature-root",
        MEAD_DRIVE if dataset == "mead" else CREMAD_DRIVE,
        "--emotion-pool-root",
        CMET_ROOT / "audios" / "MEAD",
        "--num-samples",
        str(NUM_SAMPLES),
        "--seed",
        str(SEED),
        "--device",
        "cuda",
        "--backend",
        "persistent",
        "--expression-batch-size",
        str(EXPRESSION_BATCH_SIZE),
        "--render-batch-size",
        str(RENDER_BATCH_SIZE),
        "--fail-fast",
    ]
    if limit is not None:
        command.extend(["--limit", str(limit)])
    run_command(command)


In [ ]:
if RUN_MEAD_BENCH_SMOKE:
    run_benchmark("mead", limit=2)
if RUN_CREMAD_BENCH_SMOKE:
    run_benchmark("crema-d", limit=2)
if not RUN_MEAD_BENCH_SMOKE and not RUN_CREMAD_BENCH_SMOKE:
    print("benchmark smoke 未启用。")


In [ ]:
if RUN_MEAD_BENCH_FULL:
    run_benchmark("mead")
if RUN_CREMAD_BENCH_FULL:
    run_benchmark("crema-d")
if not RUN_MEAD_BENCH_FULL and not RUN_CREMAD_BENCH_FULL:
    print("完整 benchmark 未启用。")


In [ ]:
if RUN_BENCH_TECH_CHECK:
    for dataset in ["mead", "crema-d"]:
        video_dir = BENCHMARK_OUTPUT / "videos" / BENCHMARK_EMOTION_PROTOCOL / dataset
        if video_dir.is_dir() and any(video_dir.glob("*.mp4")):
            run_repro_script(
                "evaluate_videos_basic.py",
                "--video-dir",
                video_dir,
                "--out-csv",
                REPORT_ROOT / "benchmark_technical" / CHECKPOINT_TAG / BENCHMARK_EMOTION_PROTOCOL / f"{dataset}.csv",
                "--report",
                REPORT_ROOT / "benchmark_technical" / CHECKPOINT_TAG / BENCHMARK_EMOTION_PROTOCOL / f"{dataset}.json",
                "--strict",
            )
        else:
            print("没有可检查的视频:", video_dir)
else:
    print("benchmark 技术检查未启用。")


## 24. 论文指标汇总

基础技术检查只证明视频有效，不等于论文指标。汇总脚本要求：

- `sample_metrics.csv`：`sample_id,sync_confidence,predicted_emotion`，来自与论文一致的 SyncNet 和按 benchmark 微调的 Emotion-FAN；
- `global_metrics.json`：`aitv,fid,fvd`，来自与论文一致的采样与特征协议；
- benchmark manifest 与 progress：由上一阶段自动生成。

作者没有公开这些精确评估器和 checkpoint，所以仓库提供严格覆盖率检查与论文目标值对照，但不会伪造缺失数值。


In [ ]:
if RUN_PARTIAL_METRIC_SUMMARY:
    for dataset in ["mead", "crema-d"]:
        manifest = BENCHMARK_OUTPUT / "manifests" / BENCHMARK_EMOTION_PROTOCOL / f"{dataset}.csv"
        progress = BENCHMARK_OUTPUT / "progress" / BENCHMARK_EMOTION_PROTOCOL / f"{dataset}.jsonl"
        if manifest.is_file() and progress.is_file():
            run_repro_script(
                "evaluate_paper_metrics.py",
                "--benchmark-manifest",
                manifest,
                "--progress",
                progress,
                "--use-progress-aitv",
                "--dataset",
                dataset,
                "--emotion-protocol",
                BENCHMARK_EMOTION_PROTOCOL,
                "--experiment",
                "paper_main",
                "--out-dir",
                REPORT_ROOT / "partial_paper_metrics" / CHECKPOINT_TAG / BENCHMARK_EMOTION_PROTOCOL,
                "--allow-partial",
            )
        else:
            print("缺少 benchmark manifest/progress:", dataset)
else:
    print("部分指标汇总未启用。")


In [ ]:
EXTERNAL_METRICS_ROOT = DRIVE_ROOT / "external_paper_metrics" / CHECKPOINT_TAG / BENCHMARK_EMOTION_PROTOCOL

if RUN_STRICT_PAPER_METRICS:
    for dataset in ["mead", "crema-d"]:
        sample_metrics = require_file(
            EXTERNAL_METRICS_ROOT / f"{dataset}_sample_metrics.csv",
            f"{dataset} 逐样本论文指标",
        )
        global_metrics = require_file(
            EXTERNAL_METRICS_ROOT / f"{dataset}_global_metrics.json",
            f"{dataset} 全局论文指标",
        )
        run_repro_script(
            "evaluate_paper_metrics.py",
            "--benchmark-manifest",
            BENCHMARK_OUTPUT / "manifests" / BENCHMARK_EMOTION_PROTOCOL / f"{dataset}.csv",
            "--sample-metrics",
            sample_metrics,
            "--global-metrics",
            global_metrics,
            "--progress",
            BENCHMARK_OUTPUT / "progress" / BENCHMARK_EMOTION_PROTOCOL / f"{dataset}.jsonl",
            "--dataset",
            dataset,
            "--emotion-protocol",
            BENCHMARK_EMOTION_PROTOCOL,
            "--experiment",
            "paper_main",
            "--out-dir",
            REPORT_ROOT / "strict_paper_metrics" / CHECKPOINT_TAG / BENCHMARK_EMOTION_PROTOCOL,
            "--strict",
        )
else:
    print("严格论文指标汇总未启用。")


## 25. 复现状态总览

这格只读取产物，不启动长任务。它帮助你在汇报中准确区分“已完成、未完成、受作者未公开内容阻塞”。


In [ ]:
def report_ready(path):
    path = Path(path)
    if not path.is_file():
        return False
    try:
        return bool(json.loads(path.read_text(encoding="utf-8")).get("ready"))
    except Exception:
        return False


def prepare_report_complete(path):
    path = Path(path)
    if not path.is_file():
        return False
    try:
        counts = json.loads(path.read_text(encoding="utf-8"))["counts"]
        return counts.get("failed", 0) == 0 and counts.get("prepared", 0) + counts.get("skipped", 0) > 0
    except Exception:
        return False


def prepare_state_complete(root, dataset):
    if CROP_MODE != "official":
        return True
    path = Path(root) / ".cmet_prepare_state.json"
    if not path.is_file():
        return False
    try:
        value = json.loads(path.read_text(encoding="utf-8"))
        return (
            value.get("schema_version") == 2
            and value.get("dataset") == dataset
            and value.get("crop_mode") == CROP_MODE
            and value.get("status") == "complete"
        )
    except Exception:
        return False


def mead_public_stream_complete(path):
    path = Path(path)
    if not path.is_file():
        return False
    try:
        value = json.loads(path.read_text(encoding="utf-8"))
        return (
            value.get("status") == "complete"
            and value.get("expected_identities") == 47
            and value.get("completed_identities") == 47
        )
    except Exception:
        return False


def tar_readable(path):
    path = Path(path)
    if not path.is_file() or path.stat().st_size == 0:
        return False
    try:
        with tarfile.open(path, "r") as archive:
            return next(iter(archive), None) is not None
    except Exception:
        return False


def latest_jsonl_count(path, key):
    path = Path(path)
    if not path.is_file():
        return 0
    latest = {}
    raw = path.read_text(encoding="utf-8")
    lines = raw.splitlines()
    for line_number, line in enumerate(lines, start=1):
        if line.strip():
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                if line_number == len(lines) and not raw.endswith(("\n", "\r")):
                    break
                raise
            latest[str(row[key])] = row.get("status")
    return sum(status == "complete" for status in latest.values())


if DATA_SOURCE == "public":
    mead_preprocess_ready = (
        mead_public_stream_complete(REPORT_ROOT / "mead_public_stream_state.json")
        and prepare_state_complete(MEAD_DRIVE, "mead")
    )
    cremad_preprocess_ready = (
        prepare_report_complete(REPORT_ROOT / "prepare_cremad_public_full.json")
        and prepare_state_complete(CREMAD_DRIVE, "crema-d")
    )
else:
    mead_preprocess_ready = (
        prepare_report_complete(REPORT_ROOT / "prepare_mead_full.json")
        and prepare_state_complete(MEAD_DRIVE, "mead")
    )
    cremad_preprocess_ready = (
        prepare_report_complete(REPORT_ROOT / "prepare_cremad_full.json")
        and prepare_state_complete(CREMAD_DRIVE, "crema-d")
    )


status_rows = [
    ("环境自检", report_ready(REPORT_ROOT / "colab_environment.json")),
    ("MEAD 完整预处理状态", mead_preprocess_ready),
    ("CREMA-D 完整预处理状态", cremad_preprocess_ready),
    (
        "完整特征门禁",
        all(
            report_ready(REPORT_ROOT / name)
            for name in [
                "feature_gate_training.json",
                "feature_gate_model.json",
                "feature_gate_benchmark.json",
            ]
        ),
    ),
    ("Drive 训练缓存", tar_readable(CACHE_ARCHIVE)),
    (
        "主实验 20 万 step checkpoint",
        latest_checkpoint(f"paper_main_seed{SEED}") is not None
        and checkpoint_step(latest_checkpoint(f"paper_main_seed{SEED}")) >= 200000,
    ),
    ("13 类定性结果", latest_jsonl_count(QUALITATIVE_OUTPUT / "progress.jsonl", "emotion") == 13),
]
for label, ready in status_rows:
    print(f"{'[完成]' if ready else '[未完成]'} {label}")

for dataset, expected in [("mead", 1143), ("crema-d", 1546)]:
    complete = latest_jsonl_count(
        BENCHMARK_OUTPUT / "progress" / BENCHMARK_EMOTION_PROTOCOL / f"{dataset}.jsonl",
        "sample_id",
    )
    print(f"[进度] {dataset} ({BENCHMARK_EMOTION_PROTOCOL}): {complete}/{expected}")

print("[公开限制] 精确 FID/FVD/SyncNet/Emotion-FAN 协议与权重未由作者公开。")


## 26. 论文中目前无法由官方仓库完整执行的部分

- **PD-FGC + C-MET**：官方只留下配置/特征分支，缺少 `src/PD_FGC`、权重和推理入口。
- **Qwen2.5-Omni 音频编码器消融**：官方训练入口没有公开该编码器的数据与适配代码。
- **连续情绪编辑**：论文展示了结果，但官方仓库没有发布时间变化语义向量的生成脚本。
- **统一 baseline 对比**：EAT、EAMM、EDTalk、FLOAT 的统一数据与评估环境未公开。
- **用户研究**：论文说明 10 位参与者、随机 50 个输出，但没有发布抽样列表、界面数据和原始投票。

这些项目不是算力不足，而是公开材料缺失。获得作者补充代码、checkpoint 和协议后，可以接入本仓库的 benchmark manifest 与指标汇总层。


## 完成标准

主方法复现完成时，你应拥有：

1. 环境、预处理、特征和数据门禁 JSON 报告；
2. `paper_main_seed42` 的 20 万 step checkpoint 与 TensorBoard；
3. 两组 20 万 step 损失消融 checkpoint；
4. 13 类基础/扩展情绪定性视频；
5. MEAD 1143 条和 CREMA-D 1546 条 benchmark 输出与 JSONL 进度；
6. 技术检查 CSV，以及在获得论文评估器后生成的严格论文指标表。

遇到错误时先看仓库 `docs/troubleshooting.md`，不要连续重跑后面的长任务。
